## EDA — Class Distribution

Reads CSV datasets from `data/` and plots class distributions.  
Change `DATA_SUBDIR` below to switch between dataset variants.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

ROOT = Path("..").resolve()

# ── Configuration ──────────────────────────────────────────────────────────────
# Change to any subdirectory under ROOT/data/ to switch dataset variant
DATA_SUBDIR     = "minigpt4-classify"   # e.g. 'gpt4-openai-classify', 'deepseek'
REGRESSION_DIR  = ROOT / "data" / "gpt4-openai-regression"
# ──────────────────────────────────────────────────────────────────────────────

data_path = ROOT / "data" / DATA_SUBDIR
print(f"Classification data: {data_path}")
print(f"Regression data    : {REGRESSION_DIR}")

### Classification — class distributions

In [ ]:
csv_files = sorted([f for f in os.listdir(data_path) if f.endswith(".csv")])


def format_filename(filename):
    name = filename.replace("percept_dataset_", "").replace(".csv", "")
    parts = name.split("_")
    alpha_part = parts[0].replace("alpha", "Sigma")
    prompt_part = parts[1].upper()
    if "neg" in prompt_part:
        prompt_suffix = "Negative"
    elif "plus" in prompt_part:
        prompt_suffix = "Positive"
    else:
        prompt_suffix = ""
    prompt_num = prompt_part.replace("neg", "").replace("plus", "")
    return f"PerceptSent Dataset {alpha_part} {prompt_num} {prompt_suffix}".strip()


def get_label_mapping(filename):
    if "p5" in filename.lower():
        return {4: "Positive", 3: "SlightlyPositive", 2: "Neutral", 1: "SlightlyNegative", 0: "Negative"}
    elif "p2plus" in filename.lower():
        return {1: "Negative/SlightlyNegative", 0: "Positive/Neutral/SlightlyPositive"}
    elif "p2neg" in filename.lower():
        return {1: "Positive/SlightlyPositive", 0: "Negative/Neutral/SlightlyNegative"}
    elif "p3" in filename.lower():
        return {2: "Positive", 1: "Negative", 0: "Neutral"}
    return {}


def get_color(label):
    ll = label.lower()
    if "positive" in ll and "negative" not in ll:
        return "#90EE90" if "slightly" in ll else "#228B22"
    if "negative" in ll and "positive" not in ll:
        return "#FFA07A" if "slightly" in ll else "#DC143C"
    if "neutral" in ll:
        return "#FFD700"
    return "#808080"


fig, axes = plt.subplots(len(csv_files), 1, figsize=(10, 5 * len(csv_files)))
if len(csv_files) == 1:
    axes = [axes]

for idx, file in enumerate(csv_files):
    df = pd.read_csv(data_path / file)
    class_col = df.columns[-1]
    class_counts = df[class_col].value_counts()
    class_pct = (class_counts / len(df)) * 100
    label_map = get_label_mapping(file)
    text_labels = [label_map.get(lbl, str(lbl)) for lbl in class_counts.index]
    colors = [get_color(lbl) for lbl in text_labels]

    bars = axes[idx].bar(range(len(class_counts)), class_counts.values, color=colors)
    axes[idx].set_xticks(range(len(class_counts)))
    axes[idx].set_xticklabels(text_labels, rotation=45, ha="right")
    axes[idx].set_ylabel("Count")
    axes[idx].set_title(f"Class Distribution — {format_filename(file)}")
    axes[idx].grid(axis="y", alpha=0.3)
    for bar, count, pct in zip(bars, class_counts.values, class_pct.values):
        axes[idx].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{pct:.1f}%\n({count})",
            ha="center", va="bottom", fontsize=9,
        )

plt.tight_layout()
plt.show()

### Regression — sentiment-score distributions

In [ ]:
import matplotlib.cm as cm
import numpy as np

reg_files = sorted(
    f for f in os.listdir(REGRESSION_DIR)
    if f.startswith("percept_dataset_regression") and f.endswith(".csv")
)


def reg_title(filename):
    name = filename.replace("percept_dataset_regression_", "").replace(".csv", "")
    return f"Regression Distribution: {name.upper()}"


def scale_max(filename):
    if "p5" in filename:
        return 4.0
    if "p3" in filename:
        return 2.0
    return 1.0


cmap = cm.get_cmap("RdYlGn")
fig, axes = plt.subplots(len(reg_files), 1, figsize=(12, 6 * len(reg_files)))
if len(reg_files) == 1:
    axes = [axes]

for idx, file in enumerate(reg_files):
    df = pd.read_csv(REGRESSION_DIR / file)
    target_col = "sentiment_score" if "sentiment_score" in df.columns else "sentiment"
    val_counts = df[target_col].round(2).value_counts().sort_index()
    scores = val_counts.index.values
    counts = val_counts.values
    pcts = (counts / len(df)) * 100
    max_scale = scale_max(file)
    bar_colors = [cmap(s / max_scale) for s in scores]

    bars = axes[idx].bar(range(len(scores)), counts, color=bar_colors,
                         edgecolor="black", alpha=0.8)
    axes[idx].set_xticks(range(len(scores)))
    axes[idx].set_xticklabels([f"{s:.2f}" for s in scores])
    axes[idx].set_ylabel("Count")
    axes[idx].set_xlabel("Regression Score")
    axes[idx].set_title(reg_title(file), fontsize=14, fontweight="bold")
    axes[idx].grid(axis="y", alpha=0.3, linestyle="--")
    for bar, count, pct in zip(bars, counts, pcts):
        axes[idx].text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(counts) * 0.01,
            f"{pct:.1f}%\n({count})",
            ha="center", va="bottom", fontsize=10, fontweight="bold",
        )

plt.tight_layout()
plt.show()